# 装饰器

学习目标：编写有类型约束的类和成员装饰器，区分求值、应用、初始化及新旧语义。

前置知识：类字段和方法、this、泛型元组、函数包装与生成代码。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；strict，主例使用新装饰器语义，旧式例独立配置。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/27-decorators/。

1. [decorators.ts](scripts/27-decorators/decorators.ts)：方法包装、字段初始化及类初始化装饰器。
2. [main.ts](scripts/27-decorators/main.ts)：顺序与结果断言。
3. [legacy.ts](scripts/27-decorators/legacy.ts)：旧式参数装饰器的独立示例。
4. [tsconfig.json](scripts/27-decorators/tsconfig.json)：新语义配置。
5. [tsconfig.legacy.json](scripts/27-decorators/tsconfig.legacy.json)：旧式装饰器和元数据配置。
6. [type-errors.ts](scripts/27-decorators/type-errors.ts)：错误字段类型和新语义参数装饰器反例。

## 1 装饰器参与类的定义和初始化

装饰器是附在类或成员上的函数调用机制。TypeScript 5.0 起，不开启 experimentalDecorators 时支持新语义。本章通过 tsc 转换后交给 Node.js 执行，不假定宿主能直接解析装饰器源码。

这是对应提案和 TypeScript 的转换支持，不是 ECMA-262 第 16 版已经规定的类型标注。类型标注可被擦除，装饰器则可能生成真实包装、初始化和副作用。

以下片段来自 main.ts。

```typescript
import assert from "node:assert/strict";
import { events, trace, register, trimField } from "./decorators.js";
@register
class Greeter {
  @trimField name = " Ada ";
  @trace("outer")
  @trace("inner")
  greet(prefix: string): string { return prefix + this.name; }
}
const greeter = new Greeter();
assert.equal(greeter.greet("Hi "), "Hi Ada");
assert.equal(greeter.name, "Ada");
assert.deepEqual(events, [
  "evaluate outer", "evaluate inner", "apply inner:greet", "apply outer:greet",
  "apply field name", "class class:Greeter", "class ready Greeter", "init inner", "init outer",
  "initialize field name",
  "call outer", "call inner"
]);
console.log(events.join(" | ")); // 与上述顺序一致。
console.log(greeter.greet("Hi ")); // Hi Ada。
```

Step 1：检查本章正常项目。

```bash
npm run check:27
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:27
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:27
# 事件顺序断言通过，最后输出 Hi Ada。
```

## 2 方法包装保留类型关系

trace 先接收日志标签，再返回方法装饰器。This 表示原方法接收者类型，Args 是参数元组类型，Return 是返回值类型。包装保留三者关系，不用 any 抹去方法契约。

context.name 可能是字符串或 symbol；kind、static、private 描述目标种类、静态性和私有性。context.access 提供相应成员的访问操作，具体能力随成员种类变化。addInitializer 注册稍后执行的回调，不在注册处立即执行。

method.apply(this, args) 保留调用接收者。若把方法当裸函数调用，包装仍需要正确的 this，trace 没有自动绑定实例。

以下片段来自 decorators.ts。

```typescript
export const events: string[] = [];
export function trace(label: string) {
  events.push("evaluate " + label);
  return function <This, Args extends unknown[], Return>(
    method: (this: This, ...args: Args) => Return,
    context: ClassMethodDecoratorContext<This, (this: This, ...args: Args) => Return>
  ) {
    events.push("apply " + label + ":" + String(context.name));
    context.addInitializer(function () { events.push("init " + label); });
    return function (this: This, ...args: Args): Return {
      events.push("call " + label);
      return method.apply(this, args);
    };
  };
}
```

## 3 类与字段装饰器的差别

类装饰器接收构造函数和类上下文，本例不替换类，只记录类就绪。其初始化回调里的 this 对应类；实例方法初始化回调里的 this 对应实例。

字段装饰器的首个参数是 undefined，可返回接收字段初始值的函数。本例只接受 string 并返回 trim 结果；这不是在类定义时直接读取每个实例的字段。getter、setter 和自动访问器各有相应上下文及替换规则，不能直接套用方法包装签名。

```typescript
export function register(value: Function, context: ClassDecoratorContext) {
  events.push("class " + context.kind + ":" + (context.name ?? "anonymous"));
  context.addInitializer(function () { events.push("class ready " + this.name); });
}
export function trimField(_value: undefined, context: ClassFieldDecoratorContext<unknown, string>) {
  events.push("apply field " + String(context.name));
  return function (initial: string): string {
    events.push("initialize field " + String(context.name));
    return initial.trim();
  };
}
```

## 4 四个不同的执行时点

同一成员上，装饰器表达式按书写顺序求值，应用则从下往上组合。例子先 evaluate outer 再 evaluate inner，接着 apply inner 再 apply outer。调用包装方法时先进入 outer，再进入 inner。

实例方法的初始化回调在实例字段初始化之前运行，本例依次记录 init inner 和 init outer。字段装饰器的应用属于类定义处理，其返回的字段初始化函数要等创建实例。因此，apply field name 在类定义阶段出现，initialize field name 在 init inner、init outer 之后、方法调用之前出现。main.ts 对完整序列作断言。

装饰器可以执行任意 JavaScript；类型检查不保证副作用无害，也不保证包装保持原方法全部行为。需要同时检查生成代码和目标类的实际运行。

## 5 旧式参数装饰器与元数据

experimentalDecorators 启用旧式语义，函数参数和输出与新语义不同。旧式参数装饰器接收目标、成员键和参数索引，本例记录 greet 第 0 个参数，单独编译运行。

emitDecoratorMetadata 属于旧式体系，输出包含 design:type、design:paramtypes 等调用。辅助函数检查 Reflect.metadata 是否存在；本例没有反射库，不宣称已存储或可读取元数据。参数装饰器本身仍能运行。

新语义不允许参数装饰器，也不与 emitDecoratorMetadata 兼容。只切换配置开关不能自动迁移已有装饰器函数。

以下片段来自 legacy.ts。

```typescript
const legacyEvents: string[] = [];
function mark(_target: object, key: string | symbol | undefined, index: number): void {
  legacyEvents.push(String(key) + ":" + index);
}
class Legacy {
  greet(@mark name: string): string { return "Hi " + name; }
}
console.log(new Legacy().greet("Ada"), legacyEvents.join(","));
export {};
```

以下片段来自 tsconfig.legacy.json。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": {
    "experimentalDecorators": true,
    "emitDecoratorMetadata": true,
    "outDir": ".legacy"
  },
  "files": [
    "legacy.ts"
  ]
}
```

Step 1：生成旧式对照代码。

```bash
npm run build:27:legacy
# .legacy/legacy.js 包含 __param 与 __metadata。
```

Step 2：运行旧式参数装饰器。

```bash
npm run run:27:legacy
# Hi Ada greet:0。
```

## 6 类型和装饰位置的限制

数值字段不能接受字符串初始化函数：上下文的访问类型和返回初始化函数类型都不匹配。新语义的参数 @parameter 则属于非法位置，两个问题分别来自类型契约和语法位置。

以下片段来自 type-errors.ts。

```typescript
import { trimField } from "./decorators.js";
class Invalid {
  @trimField count = 1; // TS1240、TS1270：字段类型不匹配。
}
function parameter(_value: unknown, _context: unknown) {}
class Parameters {
  method(@parameter value: string) { return value; }
}
```

Step 1：检查独立装饰器反例。

```bash
npm run errors:27
# 退出 1；字段处 TS1240、TS1270，参数处 TS1206。
```

## 本章小结

- 新旧装饰器是不同的调用与输出契约。
- 求值、应用、初始化、调用的时点需要分别观察。
- 泛型包装保留 this、参数和结果，字段初始化器要匹配字段类型。

## 练习

1. 调换 outer 与 inner 的书写顺序，先预测再更新事件断言，核对求值、应用、调用三段变化。
2. 增加第二个字符串字段使用 trimField，测试纯空白和已规范字符串。
3. 对照两套生成物，指出哪套包含参数装饰器辅助函数，说明为何没有反射库就不能宣称旧式元数据可读取。

## 参考与引用来源

- TypeScript 官方文档：[5.0 / Decorators](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-0.html#decorators) 的 addInitializer、逆序组合、新旧差异和 Well-Typed Decorators；[emitDecoratorMetadata](https://www.typescriptlang.org/tsconfig/emitDecoratorMetadata.html)：旧式元数据输出。
- GitHub（TC39）：[Class fields](https://github.com/tc39/proposal-decorators#class-fields)、[Adding initialization logic with addInitializer](https://github.com/tc39/proposal-decorators#adding-initialization-logic-with-addinitializer)：字段初始化函数及不同装饰对象的初始化时点；本章采用 TypeScript 5.0 起支持的新装饰器语义。